<a href="https://colab.research.google.com/github/SaadAh-mad/speaker-verification-experiments/blob/master/24kto40ktraining%2Bevaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#IMPORTS+CLASSES+Dataset Creation

# Install required libraries
!pip install -q transformers datasets soundfile accelerate scikit-learn

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
import os
import gc
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
from datasets import load_dataset
import datasets
import io
import soundfile as sf

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Checkpoint paths
checkpoint_dir = "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "/content"
checkpoint_path = os.path.join(checkpoint_dir, "best_voxceleb1_wavlm_astp_aam.pth")

# Split configuration: 1251 total speakers in VoxCeleb1
EVAL_SPEAKERS = set(f"id{i}" for i in range(10001, 10041))        # 40 speakers
TRAIN_SPEAKERS = sorted([f"id{i}" for i in range(10041, 11252)])  # 1211 speakers
spk2idx = {spk: idx for idx, spk in enumerate(TRAIN_SPEAKERS)}
num_speakers = len(TRAIN_SPEAKERS)

def get_speaker_id(key):
    parts = key.split("/")
    if len(parts) >= 3:
        return parts[2]
    return None

#################################################
# ASTP Pooling
#################################################
class ASTP(nn.Module):
    def __init__(self, in_dim, bottleneck_dim=128):
        super().__init__()
        self.linear1 = nn.Conv1d(in_dim, bottleneck_dim, kernel_size=1)
        self.linear2 = nn.Conv1d(bottleneck_dim, in_dim, kernel_size=1)

    def forward(self, x):
        alpha = torch.tanh(self.linear1(x))
        alpha = torch.softmax(self.linear2(alpha), dim=2)
        mean = torch.sum(alpha * x, dim=2)
        var = torch.sum(alpha * (x ** 2), dim=2) - mean ** 2
        std = torch.sqrt(var.clamp(min=1e-7))
        return torch.cat([mean, std], dim=1)

#################################################
# AAM-SoftMax
#################################################
class AAMSoftmax(nn.Module):
    def __init__(self, embedding_dim, num_classes, margin=0.2, scale=30):
        super().__init__()
        self.margin = margin
        self.scale = scale
        self.weight = nn.Parameter(torch.randn(num_classes, embedding_dim))
        nn.init.xavier_normal_(self.weight)

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        weight = F.normalize(self.weight, dim=1)
        cosine = F.linear(embeddings, weight)

        theta = torch.acos(torch.clamp(cosine, -1 + 1e-7, 1 - 1e-7))
        target_cosine = torch.cos(theta + self.margin)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1)

        logits = (one_hot * target_cosine + (1 - one_hot) * cosine)
        logits *= self.scale
        return logits

#################################################
# Model
#################################################
class WavLMSpeakerModel(nn.Module):
    def __init__(self, num_speakers):
        super().__init__()
        self.wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base")
        self.layer_weights = nn.Parameter(torch.ones(13))

        for p in self.wavlm.parameters():
            p.requires_grad = False

        self.pooling = ASTP(768)
        self.aam = AAMSoftmax(embedding_dim=1536, num_classes=num_speakers)

    def forward(self, input_values, labels=None):
        with torch.no_grad():
            outputs = self.wavlm(input_values=input_values, output_hidden_states=True)
        hidden_states = outputs.hidden_states

        weights = torch.softmax(self.layer_weights, dim=0)
        combined = 0
        for w, h in zip(weights, hidden_states):
            combined = combined + w * h

        combined = combined.transpose(1, 2)
        embedding = self.pooling(combined)
        embedding = F.normalize(embedding, dim=-1)

        logits = None
        if labels is not None:
            logits = self.aam(embedding, labels)

        return logits, embedding

# Processor and Collate Function
processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base")

def collate_fn(batch):
    audios = [x[0] for x in batch]
    labels = torch.tensor([x[1] for x in batch])
    features = processor(
        audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )
    return features.input_values, labels

#################################################
# Streaming Dataset Wrapper
#################################################
class VoxCelebStreamingDataset(IterableDataset):
    def __init__(self, stream, spk2idx):
        self.stream = stream
        self.spk2idx = spk2idx

    def __iter__(self):
        for sample in self.stream:
            try:
                key = sample["__key__"]
                spk_id = get_speaker_id(key)

                if spk_id not in self.spk2idx:
                    continue

                label = self.spk2idx[spk_id]

                audio_data = sample["wav"]

                if audio_data is None:
                    continue

                audio_bytes = audio_data["bytes"]

                audio, sr = sf.read(
    io.BytesIO(audio_bytes),
    dtype="float32"
)

                if audio.ndim > 1:
                    audio = audio.mean(axis=1)

                audio = np.asarray(audio, dtype=np.float32)

                if len(audio.shape) > 1:
                    audio = audio.squeeze()

                target_len = 48000

                if len(audio) > target_len:
                    start = np.random.randint(
                    0,
                    len(audio) - target_len
                )
                    audio = audio[start:start+target_len]
                else:
                    audio = np.pad(
                    audio,
                    (0, target_len-len(audio))
                )

                yield audio, label

            except Exception as e:
                print("Skipping sample:", e)
                continue


# Load & Filter Dataset
print("Loading Acouspike/Voxceleb1 dataset in streaming mode...")
hf_dataset = load_dataset("Acouspike/Voxceleb1", split="train", streaming=True)
#hf_dataset = hf_dataset.cast_column("wav", datasets.Audio(sampling_rate=16000))

# Filter out test speakers from training stream
def is_train_sample(sample):
    spk_id = get_speaker_id(sample["__key__"])
    return spk_id is not None and spk_id not in EVAL_SPEAKERS

train_stream = hf_dataset.filter(is_train_sample)
#train_stream = train_stream.shuffle(buffer_size=1000, seed=42)
dataset = VoxCelebStreamingDataset(train_stream, spk2idx)

loader = DataLoader(
    dataset,
    batch_size=8,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Loading Acouspike/Voxceleb1 dataset in streaming mode...


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

In [3]:
model = WavLMSpeakerModel(num_speakers).to(DEVICE)

model.load_state_dict(
    torch.load(checkpoint_path, map_location=DEVICE)
)

print(
    torch.softmax(
        model.layer_weights,
        dim=0
    ).detach().cpu()
)

config.json:   0%|          | 0.00/2.24k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

tensor([0.0531, 0.1025, 0.0620, 0.0605, 0.0900, 0.1278, 0.2432, 0.0624, 0.0299,
        0.0275, 0.0378, 0.0375, 0.0658])


In [10]:
# Initialize Model & Optimizer
model = WavLMSpeakerModel(num_speakers=num_speakers).to(DEVICE)
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    [
        model.layer_weights,
        *model.pooling.parameters(),
        *model.aam.parameters()
    ],
    lr=5e-4 # Additive Angular Margin is highly sensitive; keeping it stable
)

# Resuming logic
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}...")
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    print("Resumed training successfully!")

# Training loop
print("Training started...")
gc.collect()
torch.cuda.empty_cache()

running_loss = 0.0
step = 39000
num_steps_to_train = 40000  # Adjust as needed (e.g. 50k steps)

model.train()
stop_training = False

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Loading checkpoint from /content/drive/MyDrive/best_voxceleb1_wavlm_astp_aam.pth...
Resumed training successfully!
Training started...


In [11]:
for epoch in range(5):
    if stop_training:
        break
    print(f"\n--- Epoch {epoch+1} ---")
    for input_values, labels in loader:
        input_values = input_values.to(DEVICE)
        labels = labels.to(DEVICE)

        logits, embeddings = model(input_values, labels)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        step += 1

        if step % 100 == 0:
            avg_loss = running_loss / 100
            print(f"Step {step} | Avg Loss (last 100 steps) = {avg_loss:.4f}")
            running_loss = 0.0

        # Save checkpoint periodically
        if step % 1000 == 0:
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Checkpoint saved at step {step} to {checkpoint_path}")

        if step >= num_steps_to_train:
            stop_training = True
            break

# Final Save
torch.save(model.state_dict(), checkpoint_path)
print(f"Training completed! Final model saved to {checkpoint_path}")
print("Final learned layer weights:")
print(torch.softmax(model.layer_weights, dim=0).detach().cpu().numpy())


--- Epoch 1 ---
Step 39100 | Avg Loss (last 100 steps) = 11.9409
Step 39200 | Avg Loss (last 100 steps) = 9.9932
Step 39300 | Avg Loss (last 100 steps) = 10.7547
Step 39400 | Avg Loss (last 100 steps) = 10.9184
Step 39500 | Avg Loss (last 100 steps) = 10.1458
Step 39600 | Avg Loss (last 100 steps) = 11.6560
Step 39700 | Avg Loss (last 100 steps) = 10.9528
Step 39800 | Avg Loss (last 100 steps) = 11.8498
Step 39900 | Avg Loss (last 100 steps) = 12.1926
Step 40000 | Avg Loss (last 100 steps) = 11.5102
Checkpoint saved at step 40000 to /content/drive/MyDrive/best_voxceleb1_wavlm_astp_aam.pth
Training completed! Final model saved to /content/drive/MyDrive/best_voxceleb1_wavlm_astp_aam.pth
Final learned layer weights:
[0.05186473 0.08745316 0.03807944 0.03581056 0.07427432 0.16135031
 0.39104062 0.05009728 0.01756033 0.01490768 0.02013458 0.02610368
 0.03132327]


In [13]:
import shutil

shutil.copy(
    "/content/drive/MyDrive/best_voxceleb1_wavlm_astp_aam.pth",
    "/content/drive/MyDrive/wavlm_40k_baseline.pth"
)

'/content/drive/MyDrive/wavlm_40k_baseline.pth'

In [14]:
###EVALUATION###
#Evaluation script
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
from torch.utils.data import Dataset, DataLoader
import os
import gc
import glob
from transformers import (
    WavLMModel,
    Wav2Vec2FeatureExtractor
)
import random
import numpy as np
import glob
import os
from datasets import load_dataset
from collections import defaultdict
from sklearn.metrics import roc_curve
from sklearn.metrics import auc

def get_speaker_id(key):
    return key.split("/")[2]
EVAL_SPEAKERS = set(
    f"id{i}" for i in range(10001, 10041)
)


In [15]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/"
    "best_voxceleb1_wavlm_astp_aam.pth"
)

print("Device:", DEVICE)

Device: cuda


In [16]:
#################################################
# ASTP
#################################################

class ASTP(nn.Module):

    def __init__(self, in_dim, bottleneck_dim=128):
        super().__init__()

        self.linear1 = nn.Conv1d(
            in_dim,
            bottleneck_dim,
            kernel_size=1
        )

        self.linear2 = nn.Conv1d(
            bottleneck_dim,
            in_dim,
            kernel_size=1
        )

    def forward(self, x):

        alpha = torch.tanh(
            self.linear1(x)
        )

        alpha = torch.softmax(
            self.linear2(alpha),
            dim=2
        )

        mean = torch.sum(
            alpha * x,
            dim=2
        )

        var = torch.sum(
            alpha * (x ** 2),
            dim=2
        ) - mean ** 2

        std = torch.sqrt(
            var.clamp(min=1e-7)
        )

        return torch.cat(
            [mean, std],
            dim=1
        )


In [17]:
#################################################
# Model
#################################################

class WavLMSpeakerModel(nn.Module):

    def __init__(self, num_speakers):

        super().__init__()

        self.wavlm = WavLMModel.from_pretrained(
        "microsoft/wavlm-base"
)


        self.layer_weights = nn.Parameter(
            torch.ones(13)
        )

        for p in self.wavlm.parameters():
          p.requires_grad = False

        self.pooling = ASTP(768)

        #self.aam = AAMSoftmax(
    #embedding_dim=1536,
    #num_classes=num_speakers
#)
    def forward(self, input_values):

        with torch.no_grad():
          outputs = self.wavlm(
          input_values=input_values,
          output_hidden_states=True
    )
        hidden_states = outputs.hidden_states

        weights = torch.softmax(
            self.layer_weights,
            dim=0
        )

        combined = 0

        for w, h in zip(weights, hidden_states):
            combined = combined + w * h

        combined = combined.transpose(1, 2)

        embedding = self.pooling(
            combined
        )

        embedding = F.normalize(
            embedding,
            dim=-1
        )



        return None, embedding

In [18]:
model = WavLMSpeakerModel(
    num_speakers=1211
).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE
)

missing, unexpected = model.load_state_dict(
    checkpoint,
    strict=False
)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

model.eval()

print(
    torch.softmax(
        model.layer_weights,
        dim=0
    ).detach().cpu()
)

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Missing keys: []
Unexpected keys: ['aam.weight']
tensor([0.0519, 0.0875, 0.0381, 0.0358, 0.0743, 0.1614, 0.3910, 0.0501, 0.0176,
        0.0149, 0.0201, 0.0261, 0.0313])


In [22]:
processor = Wav2Vec2FeatureExtractor.from_pretrained(
    "microsoft/wavlm-base"
)
stream = load_dataset(
    "Acouspike/Voxceleb1",
    split="train",
    streaming=True
)

speaker_samples = defaultdict(list)

MAX_FILES_PER_SPEAKER = 10
MAX_STREAM_FILES = 30000

print("Collecting evaluation samples...")

for i, sample in enumerate(stream):

    if i >= MAX_STREAM_FILES:
        break

    speaker = get_speaker_id(
        sample["__key__"]
    )

    if speaker not in EVAL_SPEAKERS:
        continue

    if len(
        speaker_samples[speaker]
    ) >= MAX_FILES_PER_SPEAKER:
        continue

    speaker_samples[speaker].append(
        sample
    )

    if i % 1000 == 0:
        print(
            "Processed:",
            i,
            "files | speakers found:",
            len(speaker_samples)
        )

print(
    "Collected speakers:",
    len(speaker_samples)
)

for spk in speaker_samples:
    print(
        spk,
        len(speaker_samples[spk])
    )

Collected speakers: 10
id10038 10
id10019 10
id10039 10
id10009 10
id10031 10
id10026 10
id10010 10
id10005 10
id10006 10
id10021 10


In [23]:
speaker_embeddings = defaultdict(list)

print("Computing embeddings...")

for speaker, samples in speaker_samples.items():

    for sample in samples:

        audio = (
            sample["wav"]
            .get_all_samples()
            .data
            .squeeze(0)
            .numpy()
        )

        audio = audio[:48000]

        if len(audio) < 48000:
            audio = np.pad(
                audio,
                (0, 48000 - len(audio))
            )

        features = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt"
        )

        input_values = (
            features.input_values
            .to(DEVICE)
        )

        with torch.no_grad():

            _, embedding = model(
                input_values
            )

        speaker_embeddings[
            speaker
        ].append(
            embedding.cpu()
        )

Computing embeddings...


In [24]:
same_scores = []

for _ in range(500):

    speaker = random.choice(
        list(
            speaker_embeddings.keys()
        )
    )

    files = speaker_embeddings[
        speaker
    ]

    if len(files) < 2:
        continue

    emb1, emb2 = random.sample(
        files,
        2
    )

    score = F.cosine_similarity(
        emb1,
        emb2
    ).item()

    same_scores.append(score)

In [25]:
different_scores = []

speakers = list(
    speaker_embeddings.keys()
)

for _ in range(500):

    spk1, spk2 = random.sample(
        speakers,
        2
    )

    emb1 = random.choice(
        speaker_embeddings[spk1]
    )

    emb2 = random.choice(
        speaker_embeddings[spk2]
    )

    score = F.cosine_similarity(
        emb1,
        emb2
    ).item()

    different_scores.append(score)

In [26]:
labels = (
    [1] * len(same_scores)
    +
    [0] * len(different_scores)
)

scores = (
    same_scores
    +
    different_scores
)

#################################################
# BEST ACCURACY
#################################################

best_acc = 0
best_threshold = 0

for threshold in np.arange(
    -1.0,
    1.0,
    0.01
):

    correct_same = sum(
        s > threshold
        for s in same_scores
    )

    correct_diff = sum(
        s < threshold
        for s in different_scores
    )

    acc = (
        correct_same +
        correct_diff
    ) / (
        len(same_scores)
        +
        len(different_scores)
    )

    if acc > best_acc:
        best_acc = acc
        best_threshold = threshold

#################################################
# EER
#################################################

fpr, tpr, thresholds = roc_curve(
    labels,
    scores
)

fnr = 1 - tpr

eer_idx = np.nanargmin(
    np.abs(fnr - fpr)
)

eer = (
    fpr[eer_idx]
    +
    fnr[eer_idx]
) / 2

eer_threshold = thresholds[
    eer_idx
]

#################################################
# ROC AUC
#################################################

roc_auc = auc(
    fpr,
    tpr
)

#################################################
# PRINT
#################################################

print("\nRESULTS\n")

print(
    f"Best Threshold: "
    f"{best_threshold:.4f}"
)

print(
    f"Best Accuracy : "
    f"{best_acc:.4f}"
)

print()

print(
    f"EER           : "
    f"{eer:.4f}"
)

print(
    f"EER Threshold : "
    f"{eer_threshold:.4f}"
)

print(
    f"ROC AUC       : "
    f"{roc_auc:.4f}"
)

print()

print(
    f"Average Same Speaker Score: "
    f"{np.mean(same_scores):.4f}"
)

print(
    f"Average Different Speaker Score: "
    f"{np.mean(different_scores):.4f}"
)

print(
    f"Gap: "
    f"{np.mean(same_scores)-np.mean(different_scores):.4f}"
)

print()

print("Same Speaker Statistics")
print("Min:", np.min(same_scores))
print("Max:", np.max(same_scores))
print("Std:", np.std(same_scores))

print()

print("Different Speaker Statistics")
print("Min:", np.min(different_scores))
print("Max:", np.max(different_scores))
print("Std:", np.std(different_scores))


RESULTS

Best Threshold: 0.3300
Best Accuracy : 0.6590

EER           : 0.3430
EER Threshold : 0.3651
ROC AUC       : 0.7138

Average Same Speaker Score: 0.4051
Average Different Speaker Score: 0.2830
Gap: 0.1221

Same Speaker Statistics
Min: -0.058087971061468124
Max: 0.6878381371498108
Std: 0.14270395293323443

Different Speaker Statistics
Min: -0.19276459515094757
Max: 0.6542680859565735
Std: 0.16519964419024738


In [28]:
#####################################
# Evaluation
#####################################

#Evaluation script
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
from torch.utils.data import Dataset, DataLoader
import os
import gc
import glob
from transformers import (
    WavLMModel,
    Wav2Vec2FeatureExtractor
)
import random
import numpy as np
import glob
import os
from datasets import load_dataset

hf_dataset = load_dataset(
    "macabdul9/VoxCeleb_test"
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

#################################################
# ASTP
#################################################

class ASTP(nn.Module):

    def __init__(self, in_dim, bottleneck_dim=128):
        super().__init__()

        self.linear1 = nn.Conv1d(
            in_dim,
            bottleneck_dim,
            kernel_size=1
        )

        self.linear2 = nn.Conv1d(
            bottleneck_dim,
            in_dim,
            kernel_size=1
        )

    def forward(self, x):

        alpha = torch.tanh(
            self.linear1(x)
        )

        alpha = torch.softmax(
            self.linear2(alpha),
            dim=2
        )

        mean = torch.sum(
            alpha * x,
            dim=2
        )

        var = torch.sum(
            alpha * (x ** 2),
            dim=2
        ) - mean ** 2

        std = torch.sqrt(
            var.clamp(min=1e-7)
        )

        return torch.cat(
            [mean, std],
            dim=1
        )


#################################################
# Processor
#################################################

processor = Wav2Vec2FeatureExtractor.from_pretrained(
    "microsoft/wavlm-base"
)

def collate_fn(batch):

    audios = [x[0] for x in batch]

    labels = torch.tensor(
        [x[1] for x in batch]
    )

    features = processor(
        audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    return features.input_values, labels

#################################################
# Dataset
#################################################

class VoxCelebDataset(Dataset):

    def __init__(self, hf_split):

        self.data = hf_split

        speakers = sorted(
            list(
                set(
                    sample["speaker_id"]
                    for sample in hf_split
                )
            )
        )

        self.spk2idx = {
            spk: idx
            for idx, spk in enumerate(speakers)
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        sample = self.data[idx]

        audio = (
            sample["audio"]
            .get_all_samples()
            .data
            .squeeze(0)
            .numpy()
        )

        audio = audio[:48000]

        label = self.spk2idx[
            sample["speaker_id"]
        ]

        return audio, label

dataset = VoxCelebDataset(
    hf_dataset["test"]
)

print("Files:", len(dataset))
print("Speakers:", len(dataset.spk2idx))

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)


#################################################
# AAM-SoftMax
#################################################

class AAMSoftmax(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_classes,
        margin=0.2,
        scale=30
    ):
        super().__init__()

        self.margin = margin
        self.scale = scale

        self.weight = nn.Parameter(
            torch.randn(
                num_classes,
                embedding_dim
            )
        )

        nn.init.xavier_normal_(
            self.weight
        )

    def forward(
        self,
        embeddings,
        labels
    ):

        embeddings = F.normalize(
            embeddings,
            dim=1
        )

        weight = F.normalize(
            self.weight,
            dim=1
        )

        cosine = F.linear(
            embeddings,
            weight
        )

        theta = torch.acos(
            torch.clamp(
                cosine,
                -1 + 1e-7,
                1 - 1e-7
            )
        )

        target_cosine = torch.cos(
            theta + self.margin
        )

        one_hot = torch.zeros_like(
            cosine
        )

        one_hot.scatter_(
            1,
            labels.view(-1,1),
            1
        )

        logits = (
            one_hot * target_cosine
            + (1 - one_hot) * cosine
        )

        logits *= self.scale

        return logits

#################################################
# Model
#################################################

class WavLMSpeakerModel(nn.Module):

    def __init__(self, num_speakers):

        super().__init__()

        self.wavlm = WavLMModel.from_pretrained(
        "microsoft/wavlm-base"
)


        self.layer_weights = nn.Parameter(
            torch.ones(13)
        )

        for p in self.wavlm.parameters():
          p.requires_grad = False

        self.pooling = ASTP(768)
# self.aam not needed for evaluation
       # self.aam = AAMSoftmax(
   # embedding_dim=1536,
   # num_classes=num_speakers
#)
    def forward(self, input_values):

        with torch.no_grad():
          outputs = self.wavlm(
          input_values=input_values,
          output_hidden_states=True
    )
        hidden_states = outputs.hidden_states

        weights = torch.softmax(
            self.layer_weights,
            dim=0
        )

        combined = 0

        for w, h in zip(weights, hidden_states):
            combined = combined + w * h

        combined = combined.transpose(1, 2)

        embedding = self.pooling(
            combined
        )

        embedding = F.normalize(
            embedding,
            dim=-1
        )



        return None, embedding


#################################################
# Load Model
#################################################

model = WavLMSpeakerModel(
    num_speakers=1211
).to(DEVICE)

checkpoint = torch.load(
    "/content/drive/MyDrive/best_voxceleb1_wavlm_astp_aam.pth",
    map_location=DEVICE
)

missing, unexpected = model.load_state_dict(
    checkpoint,
    strict=False
)

print("Missing:", missing)
print("Unexpected:", unexpected)

model.eval()
print("Model loaded!")
print(
    torch.softmax(
        model.layer_weights,
        dim=0
    ).detach().cpu()
)

embeddings_cache = {}

def get_embedding_from_index(idx):
    if idx in embeddings_cache:
        return embeddings_cache[idx]

    sample = hf_dataset["test"][idx]

    audio = (
        sample["audio"]
        .get_all_samples()
        .data
        .squeeze(0)
        .numpy()
    )

    audio = audio[:48000]

    features = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    input_values = features.input_values.to(
        DEVICE
    )

    with torch.no_grad():

        _, embedding = model(
            input_values
        )
    embeddings_cache[idx] = embedding
    return embedding

speaker_files = {}

for idx in range(len(hf_dataset["test"])):

    speaker = hf_dataset["test"][idx]["speaker_id"]

    if speaker not in speaker_files:
        speaker_files[speaker] = []

    speaker_files[speaker].append(idx)





##SAME SPEAKER LOOP
same_scores = []

for _ in range(1000):

    speaker = random.choice(
        list(speaker_files.keys())
    )

    files = speaker_files[speaker]

    if len(files) < 2:
        continue

    idx1, idx2 = random.sample(
    files,
    2
)

    emb1 = get_embedding_from_index(idx1)
    emb2 = get_embedding_from_index(idx2)

    score = F.cosine_similarity(
        emb1,
        emb2
    ).item()

    same_scores.append(score)


##DIFFERENT SPEAKER LOOP
different_scores = []

speakers = list(
    speaker_files.keys()
)

for _ in range(1000):

    spk1, spk2 = random.sample(
        speakers,
        2
    )

    idx1 = random.choice(
    speaker_files[spk1]
)

    idx2 = random.choice(
    speaker_files[spk2]
)

    emb1 = get_embedding_from_index(idx1)
    emb2 = get_embedding_from_index(idx2)

    score = F.cosine_similarity(
        emb1,
        emb2
    ).item()

    different_scores.append(score)



#ACCURACY COMPUTE
best_acc = 0
best_threshold = 0

for threshold in np.arange(
    -1.0,
    1.0,
    0.01
):

    correct_same = sum(
        s > threshold
        for s in same_scores
    )

    correct_diff = sum(
        s < threshold
        for s in different_scores
    )

    acc = (
        correct_same +
        correct_diff
    ) / (
        len(same_scores)
        +
        len(different_scores)
    )

    if acc > best_acc:
        best_acc = acc
        best_threshold = threshold

print(
    "Best Threshold:",
    best_threshold
)

print(
    "Best Accuracy:",
    best_acc
)


Device: cuda
Files: 4874
Speakers: 40


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Missing: []
Unexpected: ['aam.weight']
Model loaded!
tensor([0.0519, 0.0875, 0.0381, 0.0358, 0.0743, 0.1614, 0.3910, 0.0501, 0.0176,
        0.0149, 0.0201, 0.0261, 0.0313])
Best Threshold: 0.3700000000000012
Best Accuracy: 0.6145


In [29]:
from sklearn.metrics import roc_curve
import numpy as np

labels = (
    [1] * len(same_scores)
    +
    [0] * len(different_scores)
)

scores = same_scores + different_scores

fpr, tpr, thresholds = roc_curve(
    labels,
    scores
)

fnr = 1 - tpr

eer_idx = np.nanargmin(
    np.abs(fnr - fpr)
)

eer = (
    fpr[eer_idx]
    +
    fnr[eer_idx]
) / 2

print(f"EER: {eer*100:.2f}%")
print(
    f"EER Threshold: "
    f"{thresholds[eer_idx]:.4f}"
)

EER: 39.00%
EER Threshold: 0.3554
